In [0]:
%run ../../config/utils

In [0]:
import pandas as pd
import mlflow
import sys

mlflow.set_registry_uri('databricks-uc')

In [0]:
recent_saturday = pd.Timestamp.today() - pd.offsets.Week(weekday=5)
recent_saturday_str = recent_saturday.strftime('%Y-%m-%d')
sales_model_uri = f"models:/{digital_sales_catalog}@champion"

In [0]:
sales_model = mlflow.xgboost.load_model(sales_model_uri)
base_Data_v2 = spark.table(digital_propensity_features).filter(f.col('FISCAL_WEEK_END') == recent_saturday_str)
try:
    base_Data_pd = base_Data_v2.toPandas()
    print("Data converted to Pandas")
except Exception as e:
    print("Error converting to Pandas:", e)
    sys.exit(1)

In [0]:

base_Data_pd["sales_score"] = sales_model.predict_proba(base_Data_pd[['BEFORE_LAST_Q60DAYS_BASKETSIZE',
 'L52W_MEDIAN_BASKETSIZE',
 'BEFORE_LAST_Q60DAYS_SALES',
 'BEFORE_LAST_QUARTER_BASKETSIZE',
 'BEFORE_LAST_QUARTER_SALES',
 'L52W_G4W_STDEV_SPEND',
 'LAST_60DAYS_INSTORE_BASKETSIZE',
 'LAST_60DAYS_SALES',]])[:, 1]



base_Data_pd_id = base_Data_pd[['MBRSHP_SID','FISCAL_WEEK_END','LATEST_MBRSHP_NBR','sales_score']]

In [0]:
df = spark.createDataFrame(base_Data_pd_id)
df.write.mode('overwrite').option('replaceWhere', f"FISCAL_WEEK_END = '{recent_saturday_str}'").saveAsTable(digital_sales_scores)

In [0]:
spark.sql(f"""
    DELETE FROM {digital_sales_scores} WHERE FISCAL_WEEK_END < '{(recent_saturday - pd.offsets.Day(365)).date()}'
""")

In [0]:
dbutils.library.restartPython()